# Обучаем модель цены Airbnb

Что нужно сделать: применить регрессию и предсказать цену ночевки

План:
1. Подготовка данных
2. train / val / test
3. Признаки
4. Модель и оценка модели
5. Сравнение моделей и проверка на test

### 1. Подготовка данных

> Заберем импорты и функции из прошлой части

In [1]:
from pathlib import Path

import boto3
import pandas as pd
import numpy as np
from botocore.exceptions import BotoCoreError, ClientError

from secretkeys import ACCESS_KEY, KEY_ID

In [2]:
S3_BUCKET = "mlops26-course-bucket"
S3_KEY = "prepared/train.csv"
LOCAL_PATH = Path("data/train.csv")

s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=KEY_ID,
    aws_secret_access_key=ACCESS_KEY,
    region_name="ru-central1",
)

> посмотрим что доступно на S3

In [3]:
response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix="prepared/")
[item["Key"] for item in response["Contents"]]

['prepared/', 'prepared/listings.csv', 'prepared/train.csv']

In [4]:
def read_s3_csv(bucket, key):
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(response["Body"], dtype={"listing_id": str})

> забрал функцию для чтения данных из первой тетрадки

In [5]:
def read_data(bucket, key, local_path=None):
    """Попробуйте S3, при сбое доступа или сети прочитайте локальную копию.

    1. В try вызовите read_s3_csv(bucket, key) и верните результат.
    2. Перехватите BotoCoreError, ClientError и OSError.
    3. Если local_path не передан, повторно поднимите исходную ошибку: raise.
    4. Иначе выведите тип ошибки и сообщение о переходе к локальному файлу.
    5. Прочитайте local_path через pd.read_csv.
       Верните DataFrame.

    Не используйте except Exception. Ошибки разбора CSV должны быть видны.
    Если локального файла нет, должен возникнуть FileNotFoundError.
    """
    try:
        return read_s3_csv(bucket, key)
    except (BotoCoreError, ClientError, OSError) as error:
        if local_path is None:
            raise
        print("ошибка S3:", type(error).__name__)
        print("читаем:", local_path)
        return pd.read_csv(local_path, dtype={"listing_id": str})

> посмотрим подробнее данные

In [6]:
df = read_data(S3_BUCKET, S3_KEY, LOCAL_PATH)
df.shape

(5724, 13)

In [7]:
df.dtypes

listing_id                    str
snapshot_date                 str
latitude                  float64
longitude                 float64
accommodates                int64
bathrooms                 float64
bedrooms                  float64
beds                      float64
minimum_nights            float64
neighbourhood_cleansed        str
property_type                 str
room_type                     str
price                     float64
dtype: object

In [8]:
df.isna().sum()

listing_id                 0
snapshot_date              0
latitude                   0
longitude                  0
accommodates               0
bathrooms                  6
bedrooms                   7
beds                      21
minimum_nights             1
neighbourhood_cleansed     0
property_type              0
room_type                  0
price                      0
dtype: int64

> есть пропуски в 4 полях, нужно будет сделать импутацию далее

In [9]:
df["price"].describe().round(1)

count     5724.0
mean       237.2
std        447.9
min         17.9
25%         97.0
50%        168.4
75%        273.9
max      23686.0
Name: price, dtype: float64

> сколько есть категорий

In [10]:
df[["neighbourhood_cleansed", "property_type", "room_type"]].nunique()

neighbourhood_cleansed    33
property_type             51
room_type                  4
dtype: int64

### 2. train / val / test
> поделим строки на три части: 60% train, 20% validation, 20% test

> разделим признаки и таргеты

In [11]:
X = df.drop(columns=["listing_id", "snapshot_date", "price"])
y = df["price"]
X.shape, y.shape

((5724, 10), (5724,))

> забираем 40% и делим пополам для val и test, остальное в train

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=42)
len(X_train), len(X_val), len(X_test)

(3434, 1145, 1145)

> смотрим пересечения

In [13]:
print(set(X_train.index).isdisjoint(X_val.index))
print(set(X_train.index).isdisjoint(X_test.index))
print(set(X_val.index).isdisjoint(X_test.index))
len(X_train) + len(X_val) + len(X_test) == len(X)

True
True
True


True

In [14]:
# сравниваем цены train и val
pd.DataFrame({"train": y_train.describe(), "validation": y_val.describe()}).round(1)

,train,validation
count,3434.0,1145.0
mean,245.7,228.9
std,534.4,313.9
min,19.0,21.7
25%,99.6,97.0
50%,170.2,169.0
75%,275.2,268.5
max,23686.0,5706.0


> медианы почти совпали, в train попал выброс

### 3. Признаки

> - заполним пропуски медианой и масштабируем
> - пропуски словом missing и кодируем OneHotEncoder

> разделим колонки на две удобные группы

In [15]:
# числовые и категориальные колонки
category_columns = ["neighbourhood_cleansed", "property_type", "room_type"]
numeric_columns = [column for column in X.columns if column not in category_columns]
numeric_columns

['latitude',
 'longitude',
 'accommodates',
 'bathrooms',
 'bedrooms',
 'beds',
 'minimum_nights']

> заполним пропуски и масштабируем

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

> в категориях заменим пропуски на 'missing' и сделаем OHE

In [17]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline([
    ("fill", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

> Собираем обе цепочки в ColumnTransformer

In [18]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("numbers", numeric_pipeline, numeric_columns),
    ("category", categorical_pipeline, category_columns)
], remainder="drop")

> склонируем и обучим на train для проверки

In [19]:
from sklearn.base import clone

probe_prepare = clone(preprocessor)
probe_prepare.fit(X_train)
X_val.shape, probe_prepare.transform(X_val).shape

((1145, 10), (1145, 92))

In [20]:
# какие медианы взял imputer
probe_prepare.named_transformers_["numbers"].named_steps["fill"].statistics_

array([51.51235886, -0.13761394,  3.        ,  1.        ,  1.        ,
        2.        ,  2.        ])

### 4. Модель и метрики

> обучаем первую модель

In [21]:
from sklearn.linear_model import Ridge

ridge_model = Pipeline([
    ("prepare", clone(preprocessor)),
    ("model", Ridge())
])
ridge_model.fit(X_train, y_train)
list(ridge_model.named_steps), ridge_model.named_steps["model"].coef_.shape

(['prepare', 'model'], (92,))

In [22]:
# сравним прогнозы с правильными ценами
val_pred = ridge_model.predict(X_val)
comparison = pd.DataFrame({"correct": y_val, "predicted": val_pred})
comparison.head(8).round(1)

,correct,predicted
5692,192.0,186.9
5153,571.0,15.6
4441,115.0,228.4
4355,190.5,332.8
3539,133.3,275.1
1288,247.5,289.5
4049,68.7,132.3
1657,56.4,202.1


In [23]:
# Оценим среднюю ошибку двумя способами MAE ручной и готовый из sklrean
from sklearn.metrics import mean_absolute_error

manual_mae = np.mean(np.abs(y_val - val_pred))
round(manual_mae, 1), round(mean_absolute_error(y_val, val_pred), 1)

(np.float64(113.2), 113.2)

In [24]:
# соберем в табличку бейзлайновые показатели на DummyRegressor
# и оформим вместе с метриками в табличку

from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)

def metric_row(name, model, features, answers):
    pred = model.predict(features)
    return {"name": name,
            "mae": mean_absolute_error(answers, pred),
            "rmse": root_mean_squared_error(answers, pred),
            "r2": r2_score(answers, pred)}

pd.DataFrame([
    metric_row("baseline", baseline, X_val, y_val),
    metric_row("ridge", ridge_model, X_val, y_val)
]).round(2)

,name,mae,rmse,r2
0,baseline,129.63,319.20,-0.03
1,ridge,113.18,281.96,0.19


In [25]:
# проверим на отрицательные цены
(val_pred < 0).sum(), val_pred.min().round(1)

(np.int64(19), np.float64(-152.1))

> забавно, но линейная модель считает, что иногда хозаин может и доплатить гостю за ночевку ))
> 
> Ridge лучше чем медиана на 16 фунтов по MAE, но 19 прогнозов отрицательны

### 5. Сравнение моделей и test

In [26]:
# соберем кандидатов с общей обработкой и оформим в табличку
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

candidates = {
    "ridge": clone(ridge_model),
    "neighbors_5": Pipeline([
        ("prepare", clone(preprocessor)),
        ("model", KNeighborsRegressor(n_neighbors=5))
    ]),
    "tree_depth_3": Pipeline([
        ("prepare", clone(preprocessor)),
        ("model", DecisionTreeRegressor(max_depth=3, random_state=42))
    ])
}
validation_rows = []
for name, model in candidates.items():
    model.fit(X_train, y_train)
    validation_rows.append(metric_row(name, model, X_val, y_val))
validation_results = pd.DataFrame(validation_rows)
validation_results.round(2)

,name,mae,rmse,r2
0,ridge,113.18,281.96,0.19
1,neighbors_5,126.40,495.16,-1.49
2,tree_depth_3,125.92,347.38,-0.23


> Тут у нас выигрывает ridge про всем трем метрикам, вероятно соседей и выбросы подводят дорогие выбросы, но зато нет отрицательных цен

In [27]:
# оформим в коде выбор модели так что побеждать будет по MAE на val
validation_results = validation_results.sort_values(["mae", "name"])
best_name = validation_results.iloc[0]["name"]
best_pipeline = candidates[best_name]
best_name, type(best_pipeline.named_steps["model"]).__name__

('ridge', 'Ridge')

In [28]:
# оценим на test теперь
pd.DataFrame([metric_row(best_name, best_pipeline, X_test, y_test)]).round(2)

,name,mae,rmse,r2
0,ridge,103.47,171.76,0.38


> даже круче оценка, но это потому что в test не попали выбросы как мы выяснили в начале